In [1]:
import numpy as np

In [2]:
from mini_latents.noise_model import IsotropicNoise, AnisotropicNoise

rng = np.random.default_rng(0)
X = rng.standard_normal((500, 40)) * rng.uniform(0.5, 3, 40) + 2.0
Xc = X - X.mean(0); S = Xc.T @ Xc / 500; diagS = np.diag(S).copy()

old_var = np.var(Xc, axis=0)                     # what the current initialize computes
a = AnisotropicNoise(); a.initialize(diagS)
i = IsotropicNoise();   i.initialize(diagS)
print(np.max(np.abs(a.psi - old_var) / old_var),
      np.max(np.abs(a.var_floor - 0.01 * old_var) / old_var),
      abs(i.psi - old_var.mean()) / old_var.mean())

8.084140243718326e-16 8.84202839156692e-18 0.0


In [3]:
from mini_latents.ppca_fa import geometric_nonzero_eig
from mini_latents.base import top_k_eig, top_k_eig_cov

rng = np.random.default_rng(0)
for N, d, k in [(2000, 100, 5), (100, 300, 5), (101, 100, 5)]:
    X = rng.standard_normal((N, 20)) @ rng.standard_normal((20, d)) + 0.5 * rng.standard_normal((N, d))
    Xc = X - X.mean(0); S = Xc.T @ Xc / N; trS = np.trace(S)

    # A: vs top_k_eig on Xc
    v_old, V_old, _ = top_k_eig(Xc, k)
    v_new, V_new = top_k_eig_cov(S, k)
    sub = np.linalg.norm(V_new - V_old @ (V_old.T @ V_new), 2)

    # B: vs current FA._init_W formula
    ev = np.linalg.svd(Xc, compute_uv=False) ** 2 / N
    r = int(np.sum(ev > ev[0] * max(N, d) * np.finfo(float).eps))
    g_old = np.exp(np.log(ev[:r]).mean())
    g_new = geometric_nonzero_eig(S, N, trS)

    print(N, d, np.max(np.abs(v_new - v_old) / v_old), sub, abs(g_new / g_old - 1))

2000 100 1.054923980500395e-15 2.4626477010285436e-15 1.021405182655144e-14
100 300 1.4140166076255039e-15 1.6083935092662247e-15 6.661338147750939e-16
101 100 8.46290521112648e-16 1.7317750541433612e-15 3.205657961302677e-12
